# || NEMO Workstation ||
© Konstantinos Andreadis 2024 (PhD @ Roux Lab & Salbreux Lab at UNIGE, Switzerland)

# -- Import Libraries --

In [ ]:
%load_ext autoreload
%autoreload 2

# Import custom module_scripts
from automated_scripts import nemo_0_load_img, nemo_1_preprocess_img, nemo_2_mesh_img, nemo_3_project_img, \
    nemo_4_extract_nematic, nemo_5_analyse_nematic, nemo_6_extract_defects, nemo_7_analyse_defects, nemo_8_crisscross, \
    nemo_morph_curvature, nemo_morph_thickness
from module_scripts import analysis, datahandler, visuals

# Import python essentials
import os
import numpy as np
import matplotlib.pyplot as plt

# Standard plotting parameters
plt.style.use('default')
channel_colours = {
    0: ["Greens", "green"],
    1: ["inferno", "inferno"],
    2: ["Blues", "blue"],
    3: ["Reds", "red"],
}

In [ ]:
# Initialise Napari viewer once
visuals.view_mesh([])

# -- Select Image --

In [ ]:
# ==== Choose Image ====
# [!] WINDOWS: Sometimes the r before the file path string is needed, no idea why.
img_path = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/200/Gas2.tif'

# ==== Choose Time Step and Channel ====
t_select = 0
c_select = 0

# (0) Import Image

In [ ]:
# ==== Load Image ====
img_load = nemo_0_load_img.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                show_figures=True, render=False,
                                img_colour=channel_colours[c_select][0],
                                render_colour=channel_colours[c_select][1])
if img_load is not None:
    resdata_dir, resfig_dir = datahandler.create_resdirs(img_path, ct_label=f"t={t_select}_c={c_select}")
else:
    resdata_dir = resfig_dir = None
img_raw, img_dim, img_scale, img_unit = img_load

In [ ]:
# ==== 3D Render Image ====
visuals.view_img(img_list=[img_raw], title_list=["Raw Image"], scale=img_scale)

## 5D Viewer

In [ ]:
# # ==== Import Stacks ====
# dims = analysis.load_img_dimensions(img_path)
# img_scale = analysis.load_img_scaling(img_path)
# num_timepoints = dims["T"]
# num_channels = dims["C"]
#
# video_stack = []
#
# for c_idx in range(num_channels):
#     time_slices = []
#     for t_idx in range(num_timepoints):
#         img_load = analysis.load_img_virtual(
#             path=img_path,
#             t_sel_idx=t_idx,
#             c_sel_idx=c_idx
#         )
#
#         if img_load is not None:
#             time_slices.append(img_load[0])
#     if time_slices:
#         video_stack.append(np.stack(time_slices, axis=0))
#
# # ==== 3D Render Image ====
# cmaps = [channel_colours[i][1] for i in range(len(video_stack))]
# visuals.view_img(
#     img_list=video_stack,
#     scale=img_scale,
#     color_list=cmaps
# )

# (1) Prepare Image for Meshing

In [ ]:
# ==== Blur & Threshold Image ====
blur_value = 6  #3
thresh_val = 20  #15
img_blur, img_thresh = nemo_1_preprocess_img.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                                  img_blur_val=blur_value, img_thresh_val=thresh_val, show_figures=True)

In [ ]:
# ==== Load Result ====
# img_blur = analysis.load_img_virtual(path=os.path.join(resfig_dir, "img_blurred.tiff"), t_sel_idx=0, c_sel_idx=0)[0]
# img_thresh = analysis.load_img_virtual(path=os.path.join(resfig_dir, "img_thresholded.tiff"), t_sel_idx=0, c_sel_idx=0)[
#     0]
# ==== 3D Render Result ====
visuals.view_img([img_raw, img_blur, img_thresh], scale=img_scale,
                 title_list=["Raw Image", "Blurred Image", "Thresholded Image"],
                 color_list=["Greens_r", "inferno", "Greys_r"], opacity_list=[1.0, 0.7, 0.7])

# (2) Mesh Image

In [ ]:
# ==== Mesh Image ====
marchcube_boxsize = 2
taubinsmooth_factor = 0.01
taubinsmooth_iterations = 200
found_meshes = nemo_2_mesh_img.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                    box_size=marchcube_boxsize, smooth_factor=taubinsmooth_factor,
                                    split_mode="along_z",
                                    smooth_iterations=taubinsmooth_iterations, show_figures=True, render=False)
raw_meshes, smooth_meshes, smooth_subset_meshes = found_meshes

In [ ]:
# ==== Load Result ====
# full_mesh = datahandler.load_mesh(filepath=os.path.join(resdata_dir, "full_mesh.ply"))
# inner_mesh = datahandler.load_mesh(filepath=os.path.join(resdata_dir, "inner_mesh.ply"))
# outer_mesh = datahandler.load_mesh(filepath=os.path.join(resdata_dir, "outer_mesh.ply"))
# raw_meshes = [full_mesh, inner_mesh, outer_mesh]
#
# full_mesh_smooth = datahandler.load_mesh(filepath=os.path.join(resdata_dir, "full_mesh_smooth.ply"))
# inner_mesh_smooth = datahandler.load_mesh(filepath=os.path.join(resdata_dir, "inner_mesh_smooth.ply"))
# outer_mesh_smooth = datahandler.load_mesh(filepath=os.path.join(resdata_dir, "outer_mesh_smooth.ply"))
# smooth_meshes = [full_mesh_smooth, inner_mesh_smooth, outer_mesh_smooth]
#
# full_mesh_smooth_subset = datahandler.load_mesh(filepath=os.path.join(resdata_dir, "full_mesh_smooth_subset.ply"))
# inner_mesh_smooth_subset = datahandler.load_mesh(filepath=os.path.join(resdata_dir, "inner_mesh_smooth_subset.ply"))
# outer_mesh_smooth_subset = datahandler.load_mesh(filepath=os.path.join(resdata_dir, "outer_mesh_smooth_subset.ply"))
# smooth_subset_meshes = [full_mesh_smooth_subset, inner_mesh_smooth_subset, outer_mesh_smooth_subset]
# ==== 3D Render Result ====
# visuals.view_mesh(raw_meshes, mesh_colors=["white", "red", "blue"], mesh_titles=["FULL", "INNER", "OUTER"],
#                   img=img_raw, scale=img_scale, vec_freq=100, hide_vectors=False, vec_length=10,
#                   vec_edge_width=0.2)
# visuals.view_mesh(smooth_meshes, mesh_colors=["white", "red", "blue"],
#                   mesh_titles=["FULL SMOOTH", "INNER SMOOTH", "OUTER SMOOTH"], img=img_raw, scale=img_scale,
#                   vec_freq=100, hide_vectors=False, vec_length=10, vec_edge_width=0.2)
visuals.view_mesh(smooth_subset_meshes, mesh_colors=["white", "red", "blue"],
                  mesh_titles=["FULL SMOOTH SUBSET", "INNER SMOOTH SUBSET", "OUTER SMOOTH SUBSET"], img=img_raw,
                  scale=img_scale, vec_freq=100, hide_vectors=False, vec_length=10, vec_edge_width=0.2)

# Quick Projection Preview

In [ ]:
# ==== Define Projection Range ====
mesh_name = "sampling_mesh"
sampl_mesh = datahandler.load_mesh(os.path.join(resdata_dir, f"{mesh_name}.ply"))
# sampl_mesh.invert()
trial_min = 0.0
trial_max = 50.0
trial_num = 50
trial_mode = "mean"
proj_broad = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, unit=img_unit, min_dist_per_vert=None,
                                scale=img_scale, min_dist=trial_min, max_dist=trial_max, num_dist=trial_num,
                                mode=trial_mode, show_proj=True, normalise=False)

In [ ]:
visuals.view_colored_mesh(mesh=sampl_mesh, mesh_blending="opaque",
                          vert_colors=visuals.color_scalar(proj_broad,
                                                           cmap="inferno", normalise=True), img=img_raw,
                          scale=img_scale,
                          mesh_opacity=1.0)

In [ ]:
depths = np.array([24, 25, 26, 27, 28])
proj_mode = "mean"
layer_thickness = np.round(np.min(img_scale) / 4, 2)
all_min = depths - layer_thickness
all_max = depths + layer_thickness
rescale_layer_mesh = False
print(f"Projection depths: {depths} {img_unit}")
all_num = [20 for i in range(len(all_min))]

all_proj = []
all_mesh = []
all_names = [f"{depths[i]}±{layer_thickness}{img_unit}" for i in range(len(depths))]

for i in range(len(depths)):
    print(f">> Projecting at depth {all_names[i]} !")
    proj_broad = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, unit=img_unit, min_dist_per_vert=None,
                                    scale=img_scale, min_dist=all_min[i], max_dist=all_max[i], num_dist=all_num[i],
                                    mode=proj_mode, show_proj=False, normalise=False)
    if rescale_layer_mesh:
        sampl_mesh_layer = analysis.scale_mesh(mesh=sampl_mesh, distance=depths[i])
    else:
        sampl_mesh_layer = sampl_mesh.copy()
    all_proj.append(proj_broad)
    all_mesh.append(sampl_mesh_layer)
all_blendings = ["opaque" for i in range(len(depths))]
all_cmaps = ["inferno" for _ in all_proj]
all_colors = [visuals.color_scalar(proj_iter / proj_iter.max(), cmap=all_cmaps[i]) for i, proj_iter in
              enumerate(all_proj)]
visuals.view_colored_mesh_multiple(mesh_list=all_mesh, vert_colors_list=all_colors, mesh_blending_list=all_blendings,
                                   name_list=all_names, img=img_raw, scale=img_scale)

# (3) Project Image onto Mesh

In [ ]:
mesh_name = "sampling_mesh"
scale_down_mesh = False
flip_normals = False
proj_dist_min = 27.9
proj_dist_max = 28.1
proj_dist_num = 20
proj_mode = "mean"
layer_label, layer_mesh, proj_layer = nemo_3_project_img.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                                              mesh_name=mesh_name, dist_min=proj_dist_min,
                                                              dist_max=proj_dist_max, dist_num=proj_dist_num,
                                                              scale_down_mesh=scale_down_mesh, proj_mode=proj_mode,
                                                              flip_normals=flip_normals, show_figures=True,
                                                              render=False)

In [ ]:
# layer_label = "full_mesh_subset_proj_0_to_10_um_mean"
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
visuals.view_colored_mesh(mesh=layer_mesh, mesh_blending="opaque",
                          vert_colors=visuals.color_scalar(proj_layer,
                                                           cmap="inferno", normalise=True), img=img_raw,
                          scale=img_scale,
                          mesh_opacity=1.0)

## Multi-Layer Render

In [ ]:
layer_label_list = ['inner_mesh_smooth_subset_proj_0_to_9_um_mean', 'outer_mesh_smooth_subset_proj_0_to_9_um_mean']
layer_mesh_list = []
layer_projection_list = []
for layer_label_i in layer_label_list:
    resdata_dir_layer = os.path.join(resdata_dir, layer_label_i)
    resfig_dir_layer = os.path.join(resfig_dir, layer_label_i)
    proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
    layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
    if layer_mesh is None:
        print(f"[!] Could not find mesh for layer {layer_label_i} !")
        break
    layer_mesh_list.append(layer_mesh)
    layer_projection_list.append(visuals.color_scalar(proj_layer, normalise=True, cmap="inferno"))
visuals.view_colored_mesh_multiple(mesh_list=layer_mesh_list,
                                   mesh_blending_list=["opaque" for _ in range(len(layer_mesh_list))],
                                   vert_colors_list=layer_projection_list, img=img_raw, scale=img_scale)

# (4) Extract Nematic Field

In [ ]:
found_projected_layers = [i for i in os.listdir(resdata_dir) if os.path.isdir(os.path.join(resdata_dir, i))]
print(found_projected_layers)

In [ ]:
# layer_label = 'inner_mesh_smooth_subset_proj_0_to_9_um_mean'

In [ ]:
nematic_extracted = nemo_4_extract_nematic.main(img_path=img_path,
                                                t_select=t_select, c_select=c_select,
                                                layer_label=layer_label,
                                                patch_mode="radius", patch_size=30,
                                                compute_num=7000,
                                                normal_validity_k=20, normal_validity_thresh=0.99,
                                                grid_n_2dcurve_analysis=30,
                                                debug_2dcurve_analysis=False, show_figures=True, render=False)
layer_label, layer_mesh, proj_layer, idxs_neigh, idxs_sel, directors_2dcurved = nematic_extracted

In [ ]:
# ==== Load Result ====
# layer_label = "outer_mesh_smooth_proj_0_to_15_um_mean"
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
idxs_neigh = np.load(os.path.join(resdata_dir_layer, "extraction_idxs_neigh.npz"), allow_pickle=True)["idxs_neigh"]

# ==== 3D Render Patch of 2D+ Orientation Analysis ====
# full_mesh_colors = np.array(["#FF0000" for _ in range(len(layer_mesh.vertices))])
# random_seed_idx = np.random.choice(range(len(idxs_neigh)))
# global_patch_idxs = idxs_neigh[random_seed_idx]
# full_mesh_colors[global_patch_idxs] = "#FFFF00"
# full_mesh_colors[idxs_sel[random_seed_idx]] = "#0000FF"
# visuals.view_colored_verts(
#     verts=layer_mesh.vertices,
#     colors=list(full_mesh_colors),
#     use_orig_color=True
# )

# ==== 3D Render Result of 2D+ Orientation Analysis ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved,
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True, cmap="Greys_r"),
                                    vec_length=15,
                                    vec_edge_width=1)

# (5) Analyse Nematic Order

In [ ]:
nematic_avg_size = 20.0
nematic_analysed = nemo_5_analyse_nematic.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                               layer_label=layer_label, avg_mode="radius", avg_size=nematic_avg_size)
layer_label, layer_mesh, proj_layer, directors_2dcurved, nematic_avg_label, directors_2dcurved_avg, s_2dcurv = nematic_analysed

In [ ]:
# ==== Load Result ====
# layer_label = "outer_mesh_smooth_proj_0_to_15_um_mean"
# nematic_avg_label = f"r-{nematic_avg_size}um"
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
idxs_neigh = np.load(os.path.join(resdata_dir_layer, f"{nematic_avg_label}_idxs_neigh.npz"), allow_pickle=True)[
    "idxs_neigh"]

s_2dcurv = datahandler.load_array(name=f"S-order_2dcurved_{nematic_avg_label}",
                                  folderpath=resdata_dir_layer)
directors_2dcurved_avg = datahandler.load_array(name=f"directors-avg_2dcurved_{nematic_avg_label}",
                                                folderpath=resdata_dir_layer)

# ==== 3D Render Curved Nematic Order ====
vec_length = 15
vec_edge_width = 1.0
patch_sel_idx = np.random.choice(range(len(idxs_neigh)))
patch_color = np.array(["#FF0000" for _ in range(len(directors_2dcurved))])
patch_color[idxs_neigh[patch_sel_idx]] = "#FFFF00"
patch_color[idxs_neigh[patch_sel_idx][0]] = "#0000FF"
# visuals.view_colored_verts(verts=directors_2dcurved[:, :3], colors=list(patch_color), use_orig_color=True)
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(s_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width, img=img_raw, scale=img_scale)

# (6) Extract Defects

In [ ]:
defects_extracted = nemo_6_extract_defects.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                                layer_label=layer_label, nematic_avg_label=nematic_avg_label,
                                                dist_cutoff_defect_localisation=50,
                                                max_candidates_defect_localisation=80, show_figures=True, render=False)
layer_label, layer_mesh, proj_layer, directors_2dcurved_avg, s_2dcurv, defect_coords = defects_extracted

In [ ]:
# ==== Load Result ====
# layer_label = 'outer_mesh_smooth_proj_0_to_15_um_mean'
# nematic_avg_label = 'r-40.0um'
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
s_2dcurv = datahandler.load_array(name=f"S-order_2dcurved_{nematic_avg_label}",
                                  folderpath=resdata_dir_layer)
defect_coords = datahandler.load_array("defect_coords", folderpath=resdata_dir_layer)
directors_2dcurved_avg = datahandler.load_array(name=f"directors-avg_2dcurved_{nematic_avg_label}",
                                                folderpath=resdata_dir_layer)
# ==== 3D Render Result ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, vec_edge_width=0.1,
                                    vec_colors=visuals.color_scalar(s_2dcurv, manual_vminmax=[0, 1]),
                                    marker_size=500,
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys"),
                                    markers=defect_coords,
                                    marker_colors=visuals.color_scalar(np.linspace(0, 1, len(defect_coords)),
                                                                       "Set1"))

# (7) Analyse Defects

In [ ]:
defects_analysed = nemo_7_analyse_defects.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                               layer_label=layer_label, nematic_avg_label=nematic_avg_label,
                                               topcurv_radius=50, topcurv_interpk=10, topcharge_mode="radius",
                                               topcharge_size=25.0, pol_mode="radius", pol_size=50.0,
                                               show_figures=True, render=False)
layer_label, layer_mesh, proj_layer, directors_2dcurved_avg, s_2dcurv, defect_idxs_calc, m_charge, charge_pol_linked_idxs, pol_vecfield = defects_analysed

In [ ]:
nematic_avg_label, layer_label

In [ ]:
# ==== Load Result ====
layer_label = 'outer_mesh_smooth_subset_proj_0_to_9_um_mean'
nematic_avg_label = 'r-20.0um'

resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"))
s_2dcurv = datahandler.load_array(name=f"S-order_2dcurved_{nematic_avg_label}",
                                  folderpath=resdata_dir_layer)
directors_2dcurved_avg = datahandler.load_array(name=f"directors-avg_2dcurved_{nematic_avg_label}",
                                                folderpath=resdata_dir_layer)
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
defect_idxs_calc = datahandler.load_array(name=f"top-charge_2dcurved_idxs",
                                          folderpath=resdata_dir_layer).astype(int)
m_charge = datahandler.load_array(name=f"top-charge_2dcurved",
                                  folderpath=resdata_dir_layer)
charge_pol_linked_idxs = datahandler.load_array(name=f"def-pol_2dcurved_idxs",
                                                folderpath=resdata_dir_layer).astype(int)
pol_vecfield = datahandler.load_array(name=f"def-pol_2dcurved",
                                      folderpath=resdata_dir_layer)
# ==== 3D Render Result ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="flat",
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys_r"),
                                    vec_colors=visuals.color_scalar(s_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]], vec_edge_width=0.3,
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400,
                                    marker_vectors=pol_vecfield, marker_vectors_length=20, vec_length=10,
                                    marker_vector_width=3,
                                    marker_vectors_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs],
                                                                              manual_vminmax=[-1, 1],
                                                                              cmap="rainbow"), img=img_raw,
                                    scale=img_scale)

# (8) Criss-Cross Strength

In [ ]:
found_analysed_layers = [
    (layer, f.split("_")[-1].split(".csv")[0])
    for layer in [i for i in os.listdir(resdata_dir) if os.path.isdir(os.path.join(resdata_dir, i))]
    for f in os.listdir(os.path.join(resdata_dir, layer))
    if f.startswith("S-order_2dcurved") and f.endswith(".csv")
]
print(found_analysed_layers)

In [ ]:

layer_name_1 = 'inner_mesh_smooth_subset_proj_1.9_to_2.1_um_mean'
patch_label_1 = 'r-20.0um'
layer_name_2 = 'inner_mesh_smooth_subset_proj_6.9_to_7.1_um_mean'
patch_label_2 = 'r-20.0um'
crisscross_analysed = nemo_8_crisscross.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                             layer_name_1=layer_name_1,
                                             layer_name_2=layer_name_2, patch_label_1=patch_label_1,
                                             patch_label_2=patch_label_2,
                                             show_figures=True, render=False)
layer_mesh_1, layer_mesh_2, proj_layer_1, proj_layer_2, field_1, field_2, plotted_vecfields, s_2dcurv_layer_1, s_2dcurv_layer_2, crisscross_mag = crisscross_analysed

In [ ]:
if crisscross_analysed is not None:
    import trimesh

    # visuals.view_colored_mesh_multiple(mesh_list=[layer_mesh_1, layer_mesh_2],
    #                                    vert_colors_list=[
    #                                        visuals.color_scalar(proj_layer_1, normalise=True,
    #                                                             cmap="Greens_r"),
    #                                        visuals.color_scalar(proj_layer_2, normalise=True,
    #                                                             cmap="Blues_r")])
    #
    # visuals.view_colored_mesh_dir_field(mesh=trimesh.util.concatenate(layer_mesh_1, layer_mesh_2),
    #                                     directors=plotted_vecfields,
    #                                     vec_colors=visuals.color_scalar(
    #                                         np.concatenate((s_2dcurv_layer_2, s_2dcurv_layer_1)),
    #                                         manual_vminmax=[0, 1],
    #                                         cmap="Spectral"),
    #                                     mesh_vert_colors=visuals.color_scalar(np.concatenate(
    #                                         (proj_layer_1, proj_layer_2)), normalise=True,
    #                                         cmap="Greys_r"),
    #                                     vec_edge_width=0.2, vec_length=7)
    visuals.view_colored_mesh_dir_field(mesh=layer_mesh_1, directors=plotted_vecfields,
                                        vec_colors=visuals.color_scalar(
                                            np.concatenate((np.ones(len(field_2)) * 0.5, crisscross_mag), axis=0),
                                            manual_vminmax=[0, 1], cmap="coolwarm", ),
                                        mesh_vert_colors="black",
                                        vec_edge_width=0.2, vec_length=7)

# Morphology Thickness

In [ ]:
# ==== Calculate Thicknesses ====
mesh_1_name = "inner_mesh_smooth_subset"
mesh_2_name = "outer_mesh_smooth_subset"
mesh_1, mesh_2, full_dist_vals = nemo_morph_thickness.main(img_path=img_path, t_select=t_select, c_select=c_select,
                                                           mesh_1_name=mesh_1_name,
                                                           mesh_2_name=mesh_2_name,
                                                           thickness_sampl_number=5000, interp_k=10, show_figures=True)

In [ ]:
# ==== Load Result ====
mesh_1 = datahandler.load_mesh(os.path.join(resdata_dir, f"{mesh_1_name}.ply"))
mesh_2 = datahandler.load_mesh(os.path.join(resdata_dir, f"{mesh_2_name}.ply"))
full_dist_vals = datahandler.load_array(name=f"{mesh_1_name}_VS_{mesh_2_name}_thickness", folderpath=resdata_dir)

# ==== 3D Render Result ====
visuals.view_colored_mesh_multiple([mesh_1, mesh_2],
                                   [visuals.color_scalar(full_dist_vals, normalise=True, cmap="coolwarm"),
                                    "white"], mesh_blending_list=["opaque", "translucent"],
                                   mesh_opacity_list=[1.0, 0.3], img=img_raw, scale=img_scale)

# Morphology Curvature

In [ ]:
# ==== Calculate Curvatures ====
mesh_name = 'sampling_mesh'
curv_num_samples = 2000
curv_radius = 20
mesh_curv, full_C_gauss, full_C_mean = nemo_morph_curvature.main(img_path=img_path, t_select=t_select,
                                                                 c_select=c_select,
                                                                 num_samples=curv_num_samples, radius=curv_radius,
                                                                 interp_k=10,
                                                                 mesh_name=mesh_name, flip_normals=False,
                                                                 show_figures=True, render=False)

In [ ]:
# ==== Load Result ====
# mesh_name = ""
mesh_curv = datahandler.load_mesh(os.path.join(resdata_dir, f"{mesh_name}.ply"))
# mesh_curv.invert()
full_C_gauss = datahandler.load_array(f"{mesh_name}_gauss_curv_r-{curv_radius}{img_unit}",
                                      folderpath=resdata_dir)
full_C_mean = datahandler.load_array(f"{mesh_name}_mean_curv_r-{curv_radius}{img_unit}",
                                     folderpath=resdata_dir)
# ==== 3D Render Result ====
visuals.view_colored_mesh_multiple([mesh_curv, mesh_curv],
                                   [visuals.color_scalar(full_C_gauss, normalise=True, cmap="coolwarm"),
                                    visuals.color_scalar(full_C_mean, normalise=True, cmap="Spectral")],
                                   name_list=[f"Gauss {mesh_name}", f"Mean {mesh_name}"])

# BATCH

In [ ]:
img_list = [
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/hydra/hydra-body.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/hydra/hydra-closeup.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/hydra/hydra-foot.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/hydra/hydra-head-1-1.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/hydra/hydra-head-1-2.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/hydra/hydra-head-2-1.tif',
    # '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/ovaries/ovaries-closeup.tif',
    # '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/ovaries/ovaries-interior.tif',
    # '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/simulated/cylinder_diagonal.tif',
    # '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/simulated/dumbbell_bridge.tif',
    # '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/simulated/shells_shifted.tif'
]
show_figures = False
render = False
for i, path in enumerate(img_list):
    print("||||||||||", len(img_list) - i, "left !")
    print(f"=========== [Progress {100 * np.round(i / len(img_list), 2)}%] {path} ===========")
    dims = analysis.load_img_dimensions(path)
    num_timepoints = dims["T"]
    num_channels = dims["C"]

    for t_select in range(num_timepoints):
        for c_select in range(num_channels):
            nemo_0_load_img.main(img_path=path, t_select=t_select, c_select=c_select,
                                 show_figures=show_figures,
                                 render=render, img_colour=channel_colours[c_select][0],
                                 render_colour=channel_colours[c_select][1])